In [1]:
import langgraph
from langgraph.graph import StateGraph,START,END
from typing import TypedDict
from dotenv import load_dotenv

In [2]:
load_dotenv()
import os
key = os.getenv("gemini_api")

In [3]:
import langchain_google_genai 
from langchain_google_genai import ChatGoogleGenerativeAI

model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=key,
    temperature=0.7
)

To check which models your api supports

In [23]:
import google.generativeai as genai

genai.configure(api_key=key)

for m in genai.list_models():
    print(m.name, m.supported_generation_methods)

c:\Users\HP\Downloads\seq_flow\myenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\HP\AppData\Local\Temp\ipykernel_20176\1290635044.py:1: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


models/gemini-2.5-flash ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
models/gemini-2.5-pro ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
models/gemini-2.0-flash ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
models/gemini-2.0-flash-001 ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
models/gemini-2.0-flash-lite-001 ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
models/gemini-2.0-flash-lite ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
models/gemini-2.5-flash-preview-tts ['countTokens', 'generateContent']
models/gemini-2.5-pro-preview-tts ['countTokens', 'generateContent', 'batchGenerateContent']
models/gemma-3-1b-it ['generateContent', 'countTokens']
models/gemma-3-4b-it ['generateContent', 'countTokens']
models/gemma-3-12b-it ['generateContent', 'countTokens']
models/gemma-3-

State Creation

In [5]:
class LLMState(TypedDict):
    topic:str
    outline:str
    content:str

Utility functions

In [6]:
def outline_gen(state: LLMState)->LLMState:
    topic = state['topic']
    prompt = f'Generate outline based on the following topic: {topic}'
    outline = model.invoke(prompt).content
    state['outline'] = outline

    return state

def content_gen(state: LLMState)->LLMState:
    outline = state['outline']
    prompt = f'Generate content based on the following outline: {outline}'
    content = model.invoke(prompt).content
    state['content'] = content

    return state

Graph Creation

In [7]:
graph = StateGraph(LLMState)

graph.add_node('og',outline_gen)
graph.add_node('cg',content_gen)

graph.add_edge(START,'og')
graph.add_edge('og','cg')
graph.add_edge('cg',END)

workflow= graph.compile()

In [8]:
initial_state = {'topic':'India'}
final_state = workflow.invoke(initial_state)

In [9]:
print(final_state['outline'])

Here's a comprehensive outline for the topic "India," covering its diverse aspects:

**Outline: India**

**I. Introduction**
    A. Hook: India – A land of ancient civilization, profound diversity, and rapid modernization.
    B. Geographic Context: Located in South Asia, 7th largest country by area, 2nd most populous.
    C. Thesis Statement: India's unique identity is shaped by its rich history, unparalleled cultural and linguistic diversity, democratic governance, and its emerging role as a global economic and geopolitical power, despite facing significant socio-economic challenges.

**II. Geography and Environment**
    A. Location and Borders:
        1. Indian subcontinent, sharing borders with Pakistan, China, Nepal, Bhutan, Bangladesh, Myanmar.
        2. Maritime borders with Sri Lanka, Maldives.
    B. Major Physical Features:
        1. Himalayas (North)
        2. Indo-Gangetic Plain (Fertile plains)
        3. Thar Desert (West)
        4. Deccan Plateau (Peninsular India)

In [10]:
print(final_state['content'])

## India: A Tapestry of Ancient Civilization, Profound Diversity, and Global Ambition

**I. Introduction**

India stands as a captivating land, a crucible where ancient civilization meets profound diversity and rapid modernization. Located strategically in South Asia, it commands attention as the world's 7th largest country by area and its 2nd most populous nation. From the snow-capped Himalayas to the sun-drenched coastal plains, India's geographical expanse mirrors its cultural depth. India's unique identity is profoundly shaped by its rich history, unparalleled cultural and linguistic diversity, a robust democratic governance, and its emerging role as a pivotal global economic and geopolitical power, all while navigating significant socio-economic challenges.

**II. Geography and Environment**

India forms the major part of the Indian subcontinent, sharing land borders with Pakistan, China, Nepal, Bhutan, Bangladesh, and Myanmar, and maritime borders with Sri Lanka and the Maldives.